In [ ]:
!pip install pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 56.2 MB/s eta 0:00:00


In [ ]:
!pip install nervaluate

In [ ]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=6b00eeff57f817855aa2fc5da29e2313588d0e482992a31505445a467e9a213f
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
import os, json, torch
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from transformers import AutoTokenizer, AutoModelForTokenClassification, set_seed
from pytorch_lightning.callbacks import EarlyStopping
from nervaluate import Evaluator
import pandas as pd

# Evaluate.py

In [ ]:
import os, json, torch
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from transformers import AutoTokenizer, AutoModelForTokenClassification, set_seed
from pytorch_lightning.callbacks import EarlyStopping
from nervaluate import Evaluator
import pandas as pd

class TransformerModel(pl.LightningModule):
    def __init__(self, model_name, tokenizer, lr, lr_factor, lr_patience, model_max_length, bio2tags, tag_list):
        super().__init__()
        self.validation_step_outputs = []
        self.test_step_outputs = []

        print("Loading AutoModel [{}] ...".format(model_name))
        self.tokenizer = tokenizer
        self.model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=len(bio2tags), from_flax=False)

        self.lr = lr
        self.lr_factor = lr_factor
        self.lr_patience = lr_patience
        self.model_max_length = model_max_length
        self.bio2tags = bio2tags
        self.tag_list = tag_list
        self.num_labels = len(bio2tags)

        # add pad token
        self.validate_pad_token()

    def validate_pad_token(self):
        if self.tokenizer.pad_token is not None:
            return
        if self.tokenizer.sep_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the SEP token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.sep_token
            return
        if self.tokenizer.eos_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the EOS token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.eos_token
            return
        if self.tokenizer.bos_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the BOS token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.bos_token
            return
        if self.tokenizer.cls_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the CLS token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.cls_token
            return
        raise Exception(
            "Could not detect SEP/EOS/BOS/CLS tokens, and thus could not assign a PAD token which is required.")

    def forward(self, input_ids, attention_mask, labels):
        output = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True
        )
        return output["loss"], output["logits"]

    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        loss, _ = self(input_ids, attention_mask, labels)
        return {"loss": loss}

    def validation_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        token_idx = batch["token_idx"]

        loss, logits = self(input_ids, attention_mask, labels)  # logits is [batch_size, seq_len, num_classes]

        batch_size = logits.size()[0]
        batch_pred = torch.argmax(logits.detach().cpu(), dim=-1).tolist()  # reduce to [batch_size, seq_len] as list
        batch_gold = labels.detach().cpu().tolist()  # [batch_size, seq_len] as list
        batch_token_idx = token_idx.detach().cpu().tolist()

        for batch_idx in range(batch_size):
            pred, gold, idx = batch_pred[batch_idx], batch_gold[batch_idx], batch_token_idx[batch_idx]
            y_hat, y = [], []
            for i in range(0, max(idx) + 1): # for each sentence
                pos = idx.index(i)  # find next token index and get pred and gold
                y_hat.append(pred[pos])
                y.append(gold[pos])

        self.validation_step_outputs.append({
            "loss": loss,
            "y": y,
            "y_hat": y_hat
        })

        return {
            "loss": loss,
            "y": y,
            "y_hat": y_hat
        }

    def on_validation_epoch_end(self):
        odf = pd.DataFrame(self.validation_step_outputs)

        mean_val_loss = odf["loss"].mean()
        gold, pred = [], []
        for _, row in odf.iterrows():
            gold.append([self.bio2tags[token_id] for token_id in row["y"]])
            pred.append([self.bio2tags[token_id] for token_id in row["y_hat"]])

        evaluator = Evaluator(gold, pred, tags=self.tag_list, loader="list")

        results, _ = evaluator.evaluate()
        self.log("valid/avg_loss", mean_val_loss, prog_bar=True)
        self.log("valid/ent_type", float(results["ent_type"]["f1"]))
        self.log("valid/partial", float(results["partial"]["f1"]))
        self.log("valid/strict", float(results["strict"]["f1"]), prog_bar=True)
        self.log("valid/exact", float(results["exact"]["f1"]))

        self.validation_step_outputs.clear()

    def test_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        token_idx = batch["token_idx"]

        loss, logits = self(input_ids, attention_mask, labels)  # logits is [batch_size, seq_len, num_classes]

        batch_size = logits.size()[0]
        batch_pred = torch.argmax(logits.detach().cpu(), dim=-1).tolist()  # reduce to [batch_size, seq_len] as list
        batch_gold = labels.detach().cpu().tolist()  # [batch_size, seq_len] as list
        batch_token_idx = token_idx.detach().cpu().tolist()

        for batch_idx in range(batch_size):
            pred, gold, idx = batch_pred[batch_idx], batch_gold[batch_idx], batch_token_idx[batch_idx]
            y_hat, y = [], []
            for i in range(0, max(idx) + 1):  # for each sentence
                pos = idx.index(i)  # find next token index and get pred and gold
                y_hat.append(pred[pos])
                y.append(gold[pos])

        self.test_step_outputs.append({
            "loss": loss,
            "y": y,
            "y_hat": y_hat
        })

        return {
            "loss": loss,
            "y": y,
            "y_hat": y_hat
        }

    def on_test_epoch_end(self):
        odf = pd.DataFrame(self.test_step_outputs)

        mean_val_loss = odf["loss"].mean()
        gold, pred = [], []
        for _, row in odf.iterrows():
            gold.append([self.bio2tags[token_id] for token_id in row["y"]])
            pred.append([self.bio2tags[token_id] for token_id in row["y_hat"]])

        evaluator = Evaluator(gold, pred, tags=self.tag_list, loader="list")

        results, _ = evaluator.evaluate()
        self.log("test/avg_loss", mean_val_loss, prog_bar=True)
        self.log("test/ent_type", results["ent_type"]["f1"])
        self.log("test/partial", results["partial"]["f1"])
        self.log("test/strict", results["strict"]["f1"])
        self.log("test/exact", results["exact"]["f1"])

        self.test_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW([p for p in self.parameters() if p.requires_grad], lr=self.lr, eps=1e-08)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer,
                    factor=self.lr_factor,
                    patience=self.lr_patience,
                    mode='max'
                ),
                'interval': 'epoch',
                'frequency': 1,
                'monitor': 'valid/strict',
                'strict': True,
                'name': 'learning_rate',
            },
        }

    def predict(self, input_string):
        input_ids = self.tokenizer.encode(input_string, add_special_tokens=False)

        # run the model
        output = self.model(input_ids=torch.unsqueeze(torch.LongTensor(input_ids), 0), return_dict=True)
        logits = output["logits"]

        # extract results
        indices = torch.argmax(logits.detach().cpu(), dim=-1).squeeze(dim=0).tolist()  # reduce to [batch_size, seq_len] as list

        output_ids = []

        for id in input_ids:
            output_ids.append(self.tokenizer.decode(id))

        output_classes = []
        for i in indices:
            output_classes.append(self.bio2tags[i])

        return output_ids, output_classes


class RoNecDataset(Dataset):
    def __init__(self, instances):
        self.instances = []

        # run check
        for instance in instances:
            ok = True
            if len(instance["ner_ids"]) != len(instance["tokens"]):
                print("Different length ner_tags found")
                ok = False
            else:
                for _, token in zip(instance["ner_ids"], instance["tokens"]):
                    if token.strip() == "":
                        ok = False
                        print("Empty token found")
            if ok:
                self.instances.append(instance)

    def __len__(self):
        return len(self.instances)

    def __getitem__(self, i):
        return self.instances[i]


class Collator(object):
    def __init__(self, tokenizer, max_seq_len):
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

        self.validate_pad_token()

    def validate_pad_token(self):
        if self.tokenizer.pad_token is not None:
            return
        if self.tokenizer.sep_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the SEP token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.sep_token
            return
        if self.tokenizer.eos_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the EOS token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.eos_token
            return
        if self.tokenizer.bos_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the BOS token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.bos_token
            return
        if self.tokenizer.cls_token is not None:
            print(f"\tNo PAD token detected, automatically assigning the CLS token as PAD.")
            self.tokenizer.pad_token = self.tokenizer.cls_token
            return
        raise Exception("Could not detect SEP/EOS/BOS/CLS tokens, and thus could not assign a PAD token which is required.")


    def __call__(self, input_batch):
        batch_input_ids, batch_labels, batch_attention, batch_token_idx = [], [], [], []
        max_len = 0

        for instance in input_batch:
            instance_ids, instance_labels, instance_attention, instance_token_idx = [], [], [], []

            for i in range(len(instance["tokens"])):
                subids = self.tokenizer.encode(instance["tokens"][i], add_special_tokens=False)
                sublabels = [instance["ner_ids"][i]]

                if len(subids) > 1:  # we have a word split in more than 1 subids, fill appropriately
                    filler_sublabel = sublabels[0] if sublabels[0] % 2 == 0 else sublabels[0] + 1
                    sublabels.extend([filler_sublabel] * (len(subids) - 1))

                instance_ids.extend(subids)  # extend with the number of subids
                instance_labels.extend(sublabels)  # extend with the number of subtags
                instance_token_idx.extend([i] * len(subids))  # extend with the id of the token

                assert len(subids) == len(sublabels) # check for possible errors in the dataset

            if len(instance_ids) != len(instance_labels):
                print(len(instance_ids))
                print(len(instance_labels))
                print(instance_ids)
                print(instance_labels)
            assert len(instance_ids) == len(instance_labels)

            # cut to max sequence length, if needed
            if len(instance_ids) > self.max_seq_len - 2:
                instance_ids = instance_ids[:self.max_seq_len - 2]
                instance_labels = instance_labels[:self.max_seq_len - 2]
                instance_token_idx = instance_token_idx[:self.max_seq_len - 2]

            # prepend and append special tokens, if needed
            if self.tokenizer.cls_token_id and self.tokenizer.sep_token_id:
                instance_ids = [self.tokenizer.cls_token_id] + instance_ids + [self.tokenizer.sep_token_id]
                instance_labels = [0] + instance_labels + [0]
                instance_token_idx = [-1] + instance_token_idx  # no need to pad the last, will do so automatically at return
            instance_attention = [1] * len(instance_ids)

            # update max_len for later padding
            max_len = max(max_len, len(instance_ids))

            # add to batch
            batch_input_ids.append(torch.LongTensor(instance_ids))
            batch_labels.append(torch.LongTensor(instance_labels))
            batch_attention.append(torch.LongTensor(instance_attention))
            batch_token_idx.append(torch.LongTensor(instance_token_idx))

        return {
            "input_ids": torch.nn.utils.rnn.pad_sequence(batch_input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id if self.tokenizer.pad_token_id else 0),
            "attention_mask": torch.nn.utils.rnn.pad_sequence(batch_attention, batch_first=True, padding_value=0),
            "labels": torch.nn.utils.rnn.pad_sequence(batch_labels, batch_first=True, padding_value=0),
            "token_idx": torch.nn.utils.rnn.pad_sequence(batch_token_idx, batch_first=True, padding_value=-1)
        }


def run_evaluation(args):
    print("Loading data...")
    with open(args.train_file, "r", encoding="utf8") as f:
        train_data = json.load(f)
    with open(args.validation_file, "r", encoding="utf8") as f:
        validation_data = json.load(f)
    with open(args.test_file, "r", encoding="utf8") as f:
        test_data = json.load(f)

    # deduce bio2 tag mapping and simple tag list, required by nervaluate
    # deduce bio2 tag mapping and simple tag list, required by nervaluate
    tags = ["O"] * 16  # tags without the B- or I- prefix
    bio2tags = ["O"] * 31  # tags with the B- and I- prefix, all tags are here

    for instance in train_data:
        for tag, tag_index in zip(instance["ner_tags"], instance["ner_ids"]):
            bio2tags[tag_index] = tag  # put the bio2 tag in its correct position
            if tag_index % 2 == 0 and tag_index > 0:
                tags[int(tag_index / 2)] = tag[2:]

    print(f"\tDataset contains {len(bio2tags)} BIO2 classes: {bio2tags}.")
    print(f"\tThere are {len(tags)} classes: {tags}\n")

    # init tokenizer and start loading data
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, strip_accents=False)
    train_dataset = RoNecDataset(train_data)
    val_dataset = RoNecDataset(validation_data)
    test_dataset = RoNecDataset(test_data)

    collator = Collator(tokenizer=tokenizer, max_seq_len=args.model_max_length)

    train_dataloader = DataLoader(train_dataset, batch_size=args.batch_size, num_workers=4, shuffle=True,
                                  collate_fn=collator, pin_memory=True)
    val_dataloader = DataLoader(val_dataset, batch_size=args.batch_size, num_workers=4, shuffle=False,
                                collate_fn=collator, pin_memory=True)
    test_dataloader = DataLoader(test_dataset, batch_size=args.batch_size, num_workers=4, shuffle=False,
                                 collate_fn=collator, pin_memory=True)

    print("\tTrain dataset has {} instances.".format(len(train_dataset)))
    print("\tValid dataset has {} instances.".format(len(val_dataset)))
    print("\tTest dataset has {} instances.\n".format(len(test_dataset)))

    itt = 0

    valid_loss = []
    valid_ent_type = []
    valid_partial = []
    valid_strict = []
    valid_exact = []
    test_loss = []
    test_ent_type = []
    test_partial = []
    test_strict = []
    test_exact = []
    while itt < args.experiment_iterations:
        print("Running experiment {}/{}".format(itt + 1, args.experiment_iterations))

        model = TransformerModel(
            model_name=args.model_name,
            lr=args.lr,
            lr_factor=args.lr_factor,
            lr_patience=args.lr_patience,
            model_max_length=args.model_max_length,
            bio2tags=bio2tags,
            tokenizer=tokenizer,
            tag_list=tags,
        )

        early_stop = EarlyStopping(
            monitor='valid/strict',
            min_delta=0.0001,
            patience=5,
            verbose=True,
            mode='max'
        )

        lr_monitor = pl.callbacks.LearningRateMonitor(logging_interval='epoch')

        model_checkpointer = pl.callbacks.ModelCheckpoint(save_top_k=1, monitor='valid/strict', dirpath=args.dirpath,
                                                          filename='{epoch}', mode='max', save_on_train_epoch_end=True)

        trainer = pl.Trainer(
            accelerator='gpu',
            devices=args.devices,
            strategy=args.strategy,
            max_epochs=args.max_epochs,
            callbacks=[lr_monitor, early_stop, model_checkpointer],
            accumulate_grad_batches=args.accumulate_grad_batches,
            gradient_clip_val=1.0,
            #limit_train_batches=50,
            #limit_val_batches=50,
        )
        trainer.fit(model, train_dataloader, val_dataloader)

        print("\nEvaluating model on the VALIDATION dataset:")
        result_valid = trainer.test(model, val_dataloader)
        print("\nEvaluating model on the TEST dataset:")
        result_test = trainer.test(model, test_dataloader)

        with open("results_ronec_{}_of_{}.json".format(itt + 1, args.experiment_iterations), "w") as f:
            json.dump(result_test[0], f, indent=4, sort_keys=True)

        valid_loss.append(result_valid[0]['test/avg_loss'])
        valid_ent_type.append(result_valid[0]['test/ent_type'])
        valid_partial.append(result_valid[0]['test/partial'])
        valid_strict.append(result_valid[0]['test/strict'])
        valid_exact.append(result_valid[0]['test/exact'])
        test_loss.append(result_test[0]['test/avg_loss'])
        test_ent_type.append(result_test[0]['test/ent_type'])
        test_partial.append(result_test[0]['test/partial'])
        test_strict.append(result_test[0]['test/strict'])
        test_exact.append(result_test[0]['test/exact'])

        itt += 1

    print("Done, writing results...\n")

    result = {
        "valid_loss": sum(valid_loss) / args.experiment_iterations,
        "valid_ent_type": sum(valid_ent_type) / args.experiment_iterations,
        "valid_partial": sum(valid_partial) / args.experiment_iterations,
        "valid_strict": sum(valid_strict) / args.experiment_iterations,
        "valid_exact": sum(valid_exact) / args.experiment_iterations,
        "test_loss": sum(test_loss) / args.experiment_iterations,
        "test_ent_type": sum(test_ent_type) / args.experiment_iterations,
        "test_partial": sum(test_partial) / args.experiment_iterations,
        "test_strict": sum(test_strict) / args.experiment_iterations,
        "test_exact": sum(test_exact) / args.experiment_iterations
    }

    with open("results_{}.json".format(args.model_name.replace("/", "_")), "w") as f:
        json.dump(result, f, indent=4, sort_keys=True)

    print("\nFinal averaged results on TEST data: ")
    from pprint import pprint
    pprint(result)


# if __name__ == "__main__":
#     from argparse import ArgumentParser

#     parser = ArgumentParser()
#     parser.add_argument('--seed', type=int, default=-1)
#     parser.add_argument('--max_epochs', type=int, default=1000)
#     parser.add_argument('--batch_size', type=int, default=8)
#     parser.add_argument('--accumulate_grad_batches', type=int, default=2)
#     parser.add_argument('--model_name', type=str, default="dumitrescustefan/bert-base-romanian-uncased-v1")
#     parser.add_argument("--train_file", type=str, default="../data/train.json")
#     parser.add_argument("--validation_file", type=str, default="../data/valid.json")
#     parser.add_argument("--test_file", type=str, default="../data/test.json")
#     parser.add_argument("--dirpath", type=str, default=None)
#     parser.add_argument('--lr', type=float, default=2e-05)
#     parser.add_argument('--lr_factor', type=float, default=2/3)
#     parser.add_argument('--lr_patience', type=float, default=5)
#     parser.add_argument('--model_max_length', type=int, default=512)
#     parser.add_argument('--experiment_iterations', type=int, default=1)
#     parser.add_argument('--devices', type=int, default=1)
#     parser.add_argument('--strategy', type=str, default=None)

#     args = parser.parse_args()

#     if args.seed >= 0:
#         pl.seed_everything(args.seed, workers=True)
#         set_seed(args.seed)
#         os.environ['PYTHONHASHSEED']=str(args.seed)
#     else:
#         print("Using a random seed.")

#     run_evaluation(args)


# Load model and dataset

In [ ]:
import torch, json
import numpy as np
from seqeval.metrics import classification_report

# ── Load best checkpoint ────────────────────────────────────
checkpoints_path = './checkpoints/'
datasets_path = './datasets/'
m_model_name = 'bert-base-multilingual-cased'
ro_model_name = 'dumitrescustefan/bert-base-romanian-cased-v1'

In [ ]:
with open(datasets_path + 'diac/train.json', "r", encoding="utf8") as f:
    train_diac = json.load(f)

In [ ]:
with open(datasets_path + 'nodiac/train.json', "r", encoding="utf8") as f:
    train_nodiac = json.load(f)

In [ ]:
tags_diac = ["O"] * 16  # tags without the B- or I- prefix
bio2tags_diac = ["O"] * 31  # tags with the B- and I- prefix, all tags are here

for instance in train_diac:
    for tag, tag_index in zip(instance["ner_tags"], instance["ner_ids"]):
        bio2tags_diac[tag_index] = tag  # put the bio2 tag in its correct position
        if tag_index % 2 == 0 and tag_index > 0:
            tags_diac[int(tag_index / 2)] = tag[2:]

In [ ]:
print(tags_diac)

['O', 'PERSON', 'ORG', 'GPE', 'LOC', 'NAT_REL_POL', 'EVENT', 'LANGUAGE', 'WORK_OF_ART', 'DATETIME', 'PERIOD', 'MONEY', 'QUANTITY', 'NUMERIC', 'ORDINAL', 'FACILITY']


In [ ]:
print(bio2tags_diac)

['O', 'B-PERSON', 'I-PERSON', 'B-ORG', 'I-ORG', 'B-GPE', 'I-GPE', 'B-LOC', 'I-LOC', 'B-NAT_REL_POL', 'I-NAT_REL_POL', 'B-EVENT', 'I-EVENT', 'B-LANGUAGE', 'I-LANGUAGE', 'B-WORK_OF_ART', 'I-WORK_OF_ART', 'B-DATETIME', 'I-DATETIME', 'B-PERIOD', 'I-PERIOD', 'B-MONEY', 'I-MONEY', 'B-QUANTITY', 'I-QUANTITY', 'B-NUMERIC', 'I-NUMERIC', 'B-ORDINAL', 'I-ORDINAL', 'B-FACILITY', 'I-FACILITY']


In [ ]:
tags_nodiac = ["O"] * 16  # tags without the B- or I- prefix
bio2tags_nodiac = ["O"] * 31  # tags with the B- and I- prefix, all tags are here

for instance in train_diac:
    for tag, tag_index in zip(instance["ner_tags"], instance["ner_ids"]):
        bio2tags_nodiac[tag_index] = tag  # put the bio2 tag in its correct position
        if tag_index % 2 == 0 and tag_index > 0:
            tags_nodiac[int(tag_index / 2)] = tag[2:]

In [ ]:
best_ckpt = checkpoints_path + 'm_diac/epoch=2.ckpt'
print(f"Best checkpoint: {best_ckpt}")

m_diac_model = TransformerModel.load_from_checkpoint(
    best_ckpt,
    weights_only=True,
    model_name=m_model_name,        # e.g., "bert-base-multilingual-cased"
    tokenizer=AutoTokenizer.from_pretrained(m_model_name),
    lr=2e-05,
    lr_factor=2/3,
    lr_patience=5,
    model_max_length=512,
    bio2tags=bio2tags_diac,
    tag_list=tags_diac
)
m_diac_model.eval()
# m_diac_model.cuda()

Best checkpoint: ./checkpoints/m_diac/epoch=2.ckpt
Loading AutoModel [bert-base-multilingual-cased] ...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

TransformerModel(
  (model): BertForTokenClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(119547, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768,

In [ ]:

best_ckpt = checkpoints_path + '/m_nodiac/epoch=3_315.ckpt'
print(f"Best checkpoint: {best_ckpt}")

m_nodiac_model = TransformerModel.load_from_checkpoint(
    best_ckpt,
    weights_only=True,
    model_name=m_model_name,
    tokenizer=AutoTokenizer.from_pretrained(m_model_name),
    lr=2e-05,
    lr_factor=2/3,
    lr_patience=5,
    model_max_length=512,
    bio2tags=bio2tags_nodiac,
    tag_list=tags_nodiac
)
m_nodiac_model.eval()
# m_nodiac_model.cuda()

Best checkpoint: ./checkpoints//m_nodiac/epoch=3_315.ckpt
Loading AutoModel [bert-base-multilingual-cased] ...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

TransformerModel(
  (model): BertForTokenClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(119547, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768,

In [ ]:
best_ckpt = checkpoints_path + 'ro_diac/epoch=1_315.ckpt'
print(f"Best checkpoint: {best_ckpt}")

ro_diac_model = TransformerModel.load_from_checkpoint(
    best_ckpt,
    weights_only=True,
    model_name=ro_model_name,
    tokenizer=AutoTokenizer.from_pretrained(ro_model_name),
    lr=2e-05,
    lr_factor=2/3,
    lr_patience=5,
    model_max_length=512,
    bio2tags=bio2tags_diac,
    tag_list=tags_diac
)
ro_diac_model.eval()
# ro_diac_model.cuda()

Best checkpoint: ./checkpoints/ro_diac/epoch=1_315.ckpt


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/397k [00:00<?, ?B/s]

Loading AutoModel [dumitrescustefan/bert-base-romanian-cased-v1] ...


model.safetensors:   0%|          | 0.00/500M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSI

TransformerModel(
  (model): BertForTokenClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(50000, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768, 

In [22]:
best_ckpt = checkpoints_path + 'ro_nodiac/epoch=5_315.ckpt'
print(f"Best checkpoint: {best_ckpt}")

ro_nodiac_model = TransformerModel.load_from_checkpoint(
    best_ckpt,
    weights_only=True,
    model_name=ro_model_name,
    tokenizer=AutoTokenizer.from_pretrained(ro_model_name),
    lr=2e-05,
    lr_factor=2/3,
    lr_patience=5,
    model_max_length=512,
    bio2tags=bio2tags_nodiac,
    tag_list=tags_nodiac
)
ro_nodiac_model.eval()
# ro_nodiac_model.cuda()

Best checkpoint: ./checkpoints/ro_nodiac/epoch=5_315.ckpt
Loading AutoModel [dumitrescustefan/bert-base-romanian-cased-v1] ...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSI

TransformerModel(
  (model): BertForTokenClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(50000, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768, 

# Crossed Inference

In [23]:
# Configs
batch_size = 8

# "robert" | "mbert"
MODEL_KEY  = "mbert"
# "diac" | "nodiac"
TRAIN_COND = "nodiac"

RUN_NAME = f"{MODEL_KEY}_train_{TRAIN_COND}"

MODEL_HUB = {
    "robert": "dumitrescustefan/bert-base-romanian-cased-v1",
    "mbert":  "bert-base-multilingual-cased",
}[MODEL_KEY]

# SEED = 315
SEED = 222

BASE     = f"/content/drive/MyDrive/rodi_study/{RUN_NAME}"
CKPT_DIR = f"./checkpoints"
RES_DIR  = f"./results"
PRED_DIR = f"./predictions"


In [ ]:
label_list = ['PERSON', 'GPE', 'LOC', 'ORG', 'LANGUAGE', 'NAT_REL_POL', 'DATETIME', 'PERIOD', 'QUANTITY', 'MONEY', 'NUMERIC', 'ORDINAL', 'FACILITY', 'WORK_OF_ART', 'EVENT']

In [24]:
def run_inference(model, dataloader, bio2tags, device="cuda"):
    """
    Mirrors the evaluate script's validation_step logic but
    stores per-sentence results instead of aggregating them.
    Returns a list of dicts: {tokens, gold_tags, pred_tags}
    """
    model.eval()
    records = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids_key, attention_mask_key, labels_key, batch_token_idx_key = batch
            input_ids      = batch[input_ids_key].to(device)
            attention_mask = batch[attention_mask_key].to(device)
            labels_tensor  = batch[labels_key].to(device)

            outputs = model(input_ids, attention_mask, labels_tensor)   # (B, seq_len, num_labels)
            logits  = outputs[1]
            preds  = torch.argmax(logits, dim=-1).cpu().numpy()
            labels = batch[labels_key].cpu().numpy()

            for sent_preds, sent_labels, token_idx in zip(preds, labels, batch[batch_token_idx_key]):
                # token_idx: list of subword positions for each word
                word_preds  = [sent_preds[idx]  for idx in token_idx]
                word_labels = [sent_labels[idx]  for idx in token_idx]

                # Decode integer ids → tag strings
                pred_tags = [bio2tags[i] for i in word_preds]
                gold_tags = [bio2tags[i] for i in word_labels]

                records.append({
                    "pred_tags": pred_tags,
                    "gold_tags": gold_tags,
                })

    return records

In [25]:
with open(datasets_path + 'diac/test.json', "r", encoding="utf8") as f:
    test_diac = json.load(f)
with open(datasets_path + 'nodiac/test.json', "r", encoding="utf8") as f:
    test_nodiac = json.load(f)

In [29]:
def eval_model(model_name, model, bio2tags_diac, bio2tags_nodiac):
    results = []
    for eval_cond, test_dataset, bio2tags in [("diac", test_diac, bio2tags_diac), ("nodiac", test_nodiac, bio2tags_nodiac)]:
        tokenizer = AutoTokenizer.from_pretrained(model_name, strip_accents=False)
        collator = Collator(tokenizer=tokenizer, max_seq_len=512)

        test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=4, shuffle=False,
                                     collate_fn=collator, pin_memory=True)


        records = run_inference(model, test_loader, bio2tags, "cpu")

        # ── Save raw predictions ────────────────────────────────
        pred_path = f"{PRED_DIR}/{RUN_NAME}_eval_{eval_cond}.json"
        with open(pred_path, "w", encoding="utf-8") as f:
            json.dump(records, f, ensure_ascii=False, indent=2)

        # ── Per-class seqeval report ────────────────────────────
        golds = [r["gold_tags"] for r in records]
        preds = [r["pred_tags"] for r in records]

        report = classification_report(golds, preds, output_dict=True, zero_division=0)
        import pandas as pd
        df = pd.DataFrame(report).T
        df.to_csv(f"{RES_DIR}/{RUN_NAME}_eval_{eval_cond}_per_class.csv")
        print(f"\n=== {RUN_NAME} → eval {eval_cond} ===")
        print(df.to_string())
        results.append((df, records))
    return tuple(results)

In [27]:
rows = []

In [31]:
MODEL_KEY  = "mbert"
TRAIN_COND = "diac"
RUN_NAME = f"{MODEL_KEY}_train_{TRAIN_COND}"

(df_diac, m_diac_diac_records), (df_nodiac, m_diac_nodiac_records) = eval_model(m_model_name, m_diac_model, bio2tags_diac, bio2tags_nodiac)

row = df_diac.loc["micro avg"] if "micro avg" in df_diac.index else df_diac.loc["weighted avg"]
rows.append({"run": 'm_diac', "eval": 'diac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})

row = df_nodiac.loc["micro avg"] if "micro avg" in df_nodiac.index else df_nodiac.loc["weighted avg"]
rows.append({"run": 'm_diac', "eval": 'nodiac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})
print(rows[-2:])


=== mbert_train_diac → eval diac ===
              precision    recall  f1-score  support
DATETIME       0.910386  0.924096  0.917190   1660.0
EVENT          0.582353  0.562500  0.572254    176.0
FACILITY       0.755319  0.533835  0.625551    133.0
GPE            0.905325  0.896484  0.900883   1536.0
LANGUAGE       0.682540  0.781818  0.728814     55.0
LOC            0.686930  0.601064  0.641135    376.0
MONEY          0.879518  0.863905  0.871642    169.0
NAT_REL_POL    0.867097  0.875000  0.871030    768.0
NUMERIC        0.947415  0.950748  0.949078   1137.0
ORDINAL        0.894928  0.678571  0.771875    364.0
ORG            0.771563  0.843773  0.806054   1357.0
PERIOD         0.835897  0.881081  0.857895    185.0
PERSON         0.859098  0.902921  0.880465   4450.0
QUANTITY       0.931937  0.962162  0.946809    185.0
WORK_OF_ART    0.601896  0.526971  0.561947    241.0
micro avg      0.857374  0.869841  0.863562  12792.0
macro avg      0.807480  0.785662  0.793508  12792.0
weighted

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/to


=== mbert_train_diac → eval nodiac ===
              precision    recall  f1-score  support
DATETIME       0.841637  0.839645  0.840640   1690.0
EVENT          0.493243  0.407821  0.446483    179.0
FACILITY       0.764045  0.485714  0.593886    140.0
GPE            0.874011  0.858161  0.866013   1544.0
LANGUAGE       0.688525  0.736842  0.711864     57.0
LOC            0.605678  0.512000  0.554913    375.0
MONEY          0.864407  0.850000  0.857143    180.0
NAT_REL_POL    0.853825  0.837802  0.845737    746.0
NUMERIC        0.934116  0.927419  0.930755   1116.0
ORDINAL        0.902834  0.652047  0.757216    342.0
ORG            0.739213  0.836324  0.784776   1393.0
PERIOD         0.661538  0.661538  0.661538    195.0
PERSON         0.828664  0.860041  0.844061   4437.0
QUANTITY       0.875000  0.943005  0.907731    193.0
WORK_OF_ART    0.597938  0.453125  0.515556    256.0
micro avg      0.821448  0.822471  0.821959  12843.0
macro avg      0.768312  0.724099  0.741221  12843.0
weight

In [32]:
MODEL_KEY  = "mbert"
TRAIN_COND = "nodiac"
RUN_NAME = f"{MODEL_KEY}_train_{TRAIN_COND}"

(df_diac, m_nodiac_diac_records), (df_nodiac, m_nodiac_nodiac_records) = eval_model(m_model_name, m_nodiac_model, bio2tags_diac, bio2tags_nodiac)

row = df_diac.loc["micro avg"] if "micro avg" in df_diac.index else df_diac.loc["weighted avg"]
rows.append({"run": 'm_nodiac', "eval": 'diac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})

row = df_nodiac.loc["micro avg"] if "micro avg" in df_nodiac.index else df_nodiac.loc["weighted avg"]
rows.append({"run": 'm_nodiac', "eval": 'nodiac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})

print(rows[-2:])

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/to


=== mbert_train_nodiac → eval diac ===
              precision    recall  f1-score  support
DATETIME       0.890117  0.873494  0.881727   1660.0
EVENT          0.628141  0.710227  0.666667    176.0
FACILITY       0.401606  0.751880  0.523560    133.0
GPE            0.894018  0.895182  0.894600   1536.0
LANGUAGE       0.605634  0.781818  0.682540     55.0
LOC            0.689119  0.707447  0.698163    376.0
MONEY          0.926136  0.964497  0.944928    169.0
NAT_REL_POL    0.890052  0.885417  0.887728    768.0
NUMERIC        0.955882  0.914688  0.934831   1137.0
ORDINAL        0.933754  0.813187  0.869310    364.0
ORG            0.818536  0.774503  0.795911   1357.0
PERIOD         0.673913  0.670270  0.672087    185.0
PERSON         0.867278  0.892809  0.879858   4450.0
QUANTITY       0.906250  0.940541  0.923077    185.0
WORK_OF_ART    0.629630  0.634855  0.632231    241.0
micro avg      0.853655  0.860929  0.857276  12792.0
macro avg      0.780671  0.814054  0.792481  12792.0
weight

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/to


=== mbert_train_nodiac → eval nodiac ===
              precision    recall  f1-score  support
DATETIME       0.888824  0.898817  0.893792   1690.0
EVENT          0.585366  0.670391  0.625000    179.0
FACILITY       0.400778  0.735714  0.518892    140.0
GPE            0.870106  0.906736  0.888043   1544.0
LANGUAGE       0.761194  0.894737  0.822581     57.0
LOC            0.686327  0.682667  0.684492    375.0
MONEY          0.956044  0.966667  0.961326    180.0
NAT_REL_POL    0.892183  0.887399  0.889785    746.0
NUMERIC        0.958487  0.931004  0.944545   1116.0
ORDINAL        0.916933  0.839181  0.876336    342.0
ORG            0.815868  0.782484  0.798827   1393.0
PERIOD         0.899497  0.917949  0.908629    195.0
PERSON         0.872592  0.898355  0.885286   4437.0
QUANTITY       0.942105  0.927461  0.934726    193.0
WORK_OF_ART    0.675000  0.632812  0.653226    256.0
micro avg      0.857197  0.872615  0.864838  12843.0
macro avg      0.808087  0.838158  0.819032  12843.0
weig

In [33]:
MODEL_KEY  = "robert"
TRAIN_COND = "diac"
RUN_NAME = f"{MODEL_KEY}_train_{TRAIN_COND}"

(df_diac, ro_diac_diac_records), (df_nodiac, ro_diac_nodiac_records) = eval_model(ro_model_name, ro_diac_model, bio2tags_diac, bio2tags_nodiac)

row = df_diac.loc["micro avg"] if "micro avg" in df_diac.index else df_diac.loc["weighted avg"]
rows.append({"run": 'ro_diac', "eval": 'diac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})

row = df_nodiac.loc["micro avg"] if "micro avg" in df_nodiac.index else df_nodiac.loc["weighted avg"]
rows.append({"run": 'ro_diac', "eval": 'nodiac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})
print(rows[-2:])

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/to


=== robert_train_diac → eval diac ===
              precision    recall  f1-score  support
DATETIME       0.856084  0.880000  0.867877   1575.0
EVENT          0.524691  0.467033  0.494186    182.0
FACILITY       0.482353  0.723529  0.578824    170.0
GPE            0.904617  0.918569  0.911540   1621.0
LANGUAGE       0.788235  0.917808  0.848101     73.0
LOC            0.706231  0.634667  0.668539    375.0
MONEY          0.908108  0.861538  0.884211    195.0
NAT_REL_POL    0.875000  0.873850  0.874425    761.0
NUMERIC        0.913880  0.943054  0.928238   1159.0
ORDINAL        0.924837  0.907051  0.915858    312.0
ORG            0.808543  0.818533  0.813507   1295.0
PERIOD         0.811927  0.867647  0.838863    204.0
PERSON         0.885184  0.924807  0.904562   4535.0
QUANTITY       0.885463  0.934884  0.909502    215.0
WORK_OF_ART    0.633858  0.572954  0.601869    281.0
micro avg      0.856455  0.879333  0.867743  12953.0
macro avg      0.793934  0.816395  0.802673  12953.0
weighte

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/to


=== robert_train_diac → eval nodiac ===
              precision    recall  f1-score  support
DATETIME       0.839652  0.875000  0.856962   1544.0
EVENT          0.513514  0.406417  0.453731    187.0
FACILITY       0.461847  0.680473  0.550239    169.0
GPE            0.907837  0.910119  0.908977   1591.0
LANGUAGE       0.753086  0.871429  0.807947     70.0
LOC            0.675141  0.651226  0.662968    367.0
MONEY          0.894180  0.880208  0.887139    192.0
NAT_REL_POL    0.867470  0.856011  0.861702    757.0
NUMERIC        0.922386  0.946354  0.934216   1193.0
ORDINAL        0.923567  0.897833  0.910518    323.0
ORG            0.800738  0.809098  0.804896   1341.0
PERIOD         0.786364  0.873737  0.827751    198.0
PERSON         0.874920  0.918363  0.896115   4471.0
QUANTITY       0.897436  0.925110  0.911063    227.0
WORK_OF_ART    0.596899  0.570370  0.583333    270.0
micro avg      0.848078  0.872403  0.860069  12900.0
macro avg      0.781002  0.804783  0.790504  12900.0
weigh

In [34]:
MODEL_KEY  = "robert"
TRAIN_COND = "nodiac"
RUN_NAME = f"{MODEL_KEY}_train_{TRAIN_COND}"

(df_diac, ro_nodiac_diac_records), (df_nodiac, ro_nodiac_nodiac_records) = eval_model(ro_model_name, ro_nodiac_model, bio2tags_diac, bio2tags_nodiac)

row = df_diac.loc["micro avg"] if "micro avg" in df_diac.index else df_diac.loc["weighted avg"]
rows.append({"run": 'ro_nodiac', "eval": 'diac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})

row = df_nodiac.loc["micro avg"] if "micro avg" in df_nodiac.index else df_nodiac.loc["weighted avg"]
rows.append({"run": 'ro_nodiac', "eval": 'nodiac',
                 "P": round(row["precision"],4),
                 "R": round(row["recall"],4),
                 "F1": round(row["f1-score"],4)})
print(rows[-2:])

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/to


=== robert_train_nodiac → eval diac ===
              precision    recall  f1-score  support
DATETIME       0.887307  0.909841  0.898433   1575.0
EVENT          0.544944  0.532967  0.538889    182.0
FACILITY       0.603352  0.635294  0.618911    170.0
GPE            0.910224  0.900679  0.905426   1621.0
LANGUAGE       0.772152  0.835616  0.802632     73.0
LOC            0.620773  0.685333  0.651458    375.0
MONEY          0.875000  0.861538  0.868217    195.0
NAT_REL_POL    0.861745  0.843627  0.852590    761.0
NUMERIC        0.940017  0.946506  0.943250   1159.0
ORDINAL        0.933555  0.900641  0.916803    312.0
ORG            0.794358  0.826255  0.809992   1295.0
PERIOD         0.895238  0.921569  0.908213    204.0
PERSON         0.897581  0.908269  0.902893   4535.0
QUANTITY       0.924779  0.972093  0.947846    215.0
WORK_OF_ART    0.636923  0.736655  0.683168    281.0
micro avg      0.865310  0.879873  0.872531  12953.0
macro avg      0.806530  0.827792  0.816581  12953.0
weigh

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/to


=== robert_train_nodiac → eval nodiac ===
              precision    recall  f1-score  support
DATETIME       0.880401  0.910622  0.895256   1544.0
EVENT          0.573964  0.518717  0.544944    187.0
FACILITY       0.573964  0.573964  0.573964    169.0
GPE            0.897307  0.900691  0.898996   1591.0
LANGUAGE       0.746988  0.885714  0.810458     70.0
LOC            0.664122  0.711172  0.686842    367.0
MONEY          0.894737  0.885417  0.890052    192.0
NAT_REL_POL    0.882353  0.871863  0.877076    757.0
NUMERIC        0.942548  0.948868  0.945698   1193.0
ORDINAL        0.931373  0.882353  0.906200    323.0
ORG            0.806025  0.818046  0.811991   1341.0
PERIOD         0.889423  0.934343  0.911330    198.0
PERSON         0.894948  0.899351  0.897144   4471.0
QUANTITY       0.911017  0.947137  0.928726    227.0
WORK_OF_ART    0.657439  0.703704  0.679785    270.0
micro avg      0.867408  0.876822  0.872089  12900.0
macro avg      0.809774  0.826131  0.817231  12900.0
wei

In [ ]:
summary = (pd.DataFrame(rows)
             .pivot_table(index="run", columns="eval", values=["P","R","F1"]))
print(summary.to_markdown())
summary.to_csv("4x4_matrix.csv")

| run       |   ('F1', 'diac') |   ('F1', 'nodiac') |   ('P', 'diac') |   ('P', 'nodiac') |   ('R', 'diac') |   ('R', 'nodiac') |
|:----------|-----------------:|-------------------:|----------------:|------------------:|----------------:|------------------:|
| m_diac    |           0.8636 |             0.822  |          0.8574 |            0.8214 |          0.8698 |            0.8225 |
| m_nodiac  |           0.8573 |             0.8648 |          0.8537 |            0.8572 |          0.8609 |            0.8726 |
| ro_diac   |           0.8677 |             0.8601 |          0.8565 |            0.8481 |          0.8793 |            0.8724 |
| ro_nodiac |           0.8725 |             0.8721 |          0.8653 |            0.8674 |          0.8799 |            0.8768 |


# Error analysis

In [36]:
def categorize_errors(records, sent_offset=0):
    """
    Takes the records list from run_inference (gold_tags, pred_tags per sentence).
    Returns a DataFrame with one row per span-level error.
    error_type ∈ {missed, spurious, wrong_type, wrong_boundary}
    """
    import pandas as pd

    def get_spans(tags):
        spans, i = {}, 0
        while i < len(tags):
            if tags[i].startswith("B-"):
                label = tags[i][2:]
                j = i + 1
                while j < len(tags) and tags[j] == f"I-{label}":
                    j += 1
                spans[(i, j - 1)] = label
                i = j
            else:
                i += 1
        return spans

    rows = []
    for sid, rec in enumerate(records, start=sent_offset):
        gold_spans = get_spans(rec["gold_tags"])
        pred_spans = get_spans(rec["pred_tags"])

        done_gold, done_pred = set(), set()

        # ─── 1. EXACT MATCHES (True Positives & Wrong Type) ───────────────────
        for pos in set(gold_spans) & set(pred_spans):
            gl, pl = gold_spans[pos], pred_spans[pos]
            if gl != pl:
                rows.append(dict(sent_id=sid, error_type="wrong_type",
                                gold_class=gl, pred_class=pl,
                                gold_span=pos, pred_span=pos))

            # Mark both as handled regardless of label match
            done_gold.add(pos)
            done_pred.add(pos)

        # ─── 2. OVERLAPPING BOUNDARIES (Wrong Boundary) ───────────────────────
        for gpos, gl in gold_spans.items():
            if gpos in done_gold:
                continue

            for ppos, pl in pred_spans.items():
                if ppos in done_pred:
                    continue

                gs, ge = gpos
                ps, pe = ppos

                # Check overlap condition: same class, boundaries intersect, not identical
                if gl == pl and gs <= pe and ps <= ge and gpos != ppos:
                    rows.append(dict(sent_id=sid, error_type="wrong_boundary",
                                    gold_class=gl, pred_class=pl,
                                    gold_span=gpos, pred_span=ppos))
                    done_gold.add(gpos)
                    done_pred.add(ppos)
                    break  # Move to next gold span since this one is resolved

        # ─── 3. CLEAN UP (Missed & Spurious) ──────────────────────────────────
        # Missed (False Negatives)
        for pos, label in gold_spans.items():
            if pos not in done_gold:
                rows.append(dict(sent_id=sid, error_type="missed",
                                gold_class=label, pred_class="O",
                                gold_span=pos, pred_span=None))

        # Spurious (False Positives)
        for pos, label in pred_spans.items():
            if pos not in done_pred:
                rows.append(dict(sent_id=sid, error_type="spurious",
                                gold_class="O", pred_class=label,
                                gold_span=None, pred_span=pos))

    return pd.DataFrame(rows)


for records, eval_cond, run_name in zip([m_diac_diac_records, m_diac_nodiac_records, m_nodiac_diac_records, m_nodiac_nodiac_records, ro_diac_diac_records, ro_diac_nodiac_records, ro_nodiac_diac_records, ro_nodiac_nodiac_records], ['diac', 'nodiac', 'diac', 'nodiac', 'diac', 'nodiac', 'diac', 'nodiac'], ['m_diac_diac', 'm_diac_nodiac', 'm_nodiac_diac', 'm_nodiac_nodiac', 'ro_diac_diac', 'ro_diac_nodiac', 'ro_nodiac_diac', 'ro_nodiac_nodiac']):

    errors = categorize_errors(records)
    errors.to_csv(f"{RES_DIR}/{run_name}_eval_{eval_cond}_errors.csv", index=False)

    print(f"\n--- {eval_cond} error breakdown ---")
    print(errors.groupby(["gold_class", "error_type"]).size().unstack(fill_value=0))


--- diac error breakdown ---
error_type   missed  spurious  wrong_boundary  wrong_type
gold_class                                               
DATETIME         72         0              46          11
EVENT            53         0              18           6
FACILITY         27         0               5          30
GPE              87         0              23          49
LANGUAGE          5         0               1           6
LOC              83         0              20          47
MONEY             6         0               9           8
NAT_REL_POL      62         0              11          23
NUMERIC          37         0               7          12
O                 0       876               0           0
ORDINAL          93         0              13          11
ORG             119         0              78          16
PERIOD           12         0               9           1
PERSON          261         0             152          27
QUANTITY          4         0             

# Attention analysis

In [38]:
def extract_attentions(model, tokenizer, sentence, device="cuda"):
    """
    Returns (token_strings, attentions_ndarray).
    attentions shape: (num_layers, num_heads, seq_len, seq_len)
    """
    model.model.set_attn_implementation("eager")
    model.model.config.output_attentions = True

    model.eval()
    enc = tokenizer(sentence, return_tensors="pt",
                    truncation=True, max_length=512).to(device)

    with torch.no_grad():
        # Pass output_attentions=True directly to the underlying BERT:
        out = model.model.bert(**enc, output_attentions=True)

    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    attns  = np.stack([a.squeeze(0).cpu().numpy() for a in out.attentions])
    return tokens, attns  # (num_layers, num_heads, seq, seq)


configuration = [
    {
        'records': m_diac_diac_records,
        'eval_cond': 'diac',
        'run_name': 'm_diac_diac',
        'model': m_diac_model,
        'model_name': m_model_name,
    },
    {
        'records': m_diac_nodiac_records,
        'eval_cond': 'nodiac',
        'run_name': 'm_diac_nodiac',
        'model': m_diac_model,
        'model_name': m_model_name,
    },
    {
        'records': m_nodiac_diac_records,
        'eval_cond': 'diac',
        'run_name': 'm_nodiac_diac',
        'model': m_nodiac_model,
        'model_name': m_model_name,
    },
    {
        'records': m_nodiac_nodiac_records,
        'eval_cond': 'nodiac',
        'run_name': 'm_nodiac_nodiac',
        'model': m_nodiac_model,
        'model_name': m_model_name,
    },
    {
        'records': ro_diac_diac_records,
        'eval_cond': 'diac',
        'run_name': 'ro_diac_diac',
        'model': ro_diac_model,
        'model_name': ro_model_name,
    },
    {
        'records': ro_diac_nodiac_records,
        'eval_cond': 'nodiac',
        'run_name': 'ro_diac_nodiac',
        'model': ro_diac_model,
        'model_name': ro_model_name,
    },
    {
        'records': ro_nodiac_diac_records,
        'eval_cond': 'diac',
        'run_name': 'ro_nodiac_diac',
        'model': ro_nodiac_model,
        'model_name': ro_model_name,
    },
    {
        'records': ro_nodiac_nodiac_records,
        'eval_cond': 'nodiac',
        'run_name': 'ro_nodiac_nodiac',
        'model': ro_nodiac_model,
        'model_name': ro_model_name,
    }
]

for config in configuration:
    tokenizer = AutoTokenizer.from_pretrained(config['model_name'], strip_accents=False)

    # Save attention for the first N missed entities from the eval:
    errors = pd.read_csv(f"{RES_DIR}/{config['run_name']}_eval_{config['eval_cond']}_errors.csv")

    sample_ids = errors[errors["error_type"] == "missed"]["sent_id"].unique()[:3]

    for sid in sample_ids:
        rec     = config['records'][int(sid)]
        sentence = " ".join(rec["gold_tags"])  # or reconstruct from raw dataset
        tokens, attns = extract_attentions(config['model'], tokenizer, sentence, "cpu")
        np.savez_compressed(
            f"{RES_DIR}/attn_sent{sid}_{config['run_name']}.npz",
            attns=attns,
            tokens=np.array(tokens, dtype=object),
        )